# 🐍 Project 3 - Snake Water Gun Game

## What it demonstrates
| Concept | Where used |
|---------|------------|
| `random` module | Computer's random choice |
| Dictionaries | Win-condition mapping, emoji map |
| OOP | `GameSession` class tracks state |
| Dataclasses | `RoundResult` stores each round |
| Enums | `Choice` as a proper enum type |
| Statistics | Win rate, streaks, most-played |
| f-strings | Colourful scoreboard |

## Rules
```
🐍 Snake  drinks  💧 Water  → Snake wins
💧 Water  drowns  🔫 Gun    → Water wins
🔫 Gun    kills   🐍 Snake  → Gun wins
```

## Game flow
```
Player picks  →  Computer picks randomly
       ↓
  Determine winner via BEATS dict
       ↓
  Update scores, streak, history
       ↓
  Print round result + running score
       ↓
  Continue until player quits → show final stats
```

In [1]:
# ============================================================
#  PROJECT 3 — SNAKE WATER GUN GAME
# ============================================================

import random
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional, List
from collections import Counter

class Choice(Enum):
    SNAKE = "snake"
    WATER = "water"
    GUN   = "gun"

# Snake beats Water, Water beats Gun, Gun beats Snake
BEATS: dict[Choice, Choice] = {
    Choice.SNAKE: Choice.WATER,
    Choice.WATER: Choice.GUN,
    Choice.GUN:   Choice.SNAKE,
}

EMOJI: dict[Choice, str] = {
    Choice.SNAKE: "🐍 Snake",
    Choice.WATER: "💧 Water",
    Choice.GUN:   "🔫 Gun",
}

WIN_MSG  = ["You crushed it!",  "Nailed it!",  "Too easy for you!", "Unstoppable!"]
LOSE_MSG = ["Computer wins!",   "Better luck!", "So close...",       "Try again!"]
TIE_MSG  = ["Great minds...",   "Snap!",        "Dead heat!",        "Mirror match!"]

@dataclass
class RoundResult:
    round_num:    int
    player:       Choice
    computer:     Choice
    outcome:      str           # 'win' | 'lose' | 'tie'

class GameSession:
    """Manages a full Snake-Water-Gun game session."""

    def __init__(self, player_name: str = "Player"):
        self.player_name  = player_name
        self.wins         = 0
        self.losses       = 0
        self.ties         = 0
        self.rounds:  List[RoundResult] = []
        self.streak       = 0          # positive = win streak, negative = lose streak
        self.best_streak  = 0

    @property
    def total_rounds(self) -> int:
        return len(self.rounds)

    @property
    def win_rate(self) -> float:
        if self.total_rounds == 0: return 0.0
        return self.wins / self.total_rounds * 100

    def _determine_outcome(self, player: Choice, computer: Choice) -> str:
        if player == computer:
            return "tie"
        elif BEATS[player] == computer:
            return "win"
        else:
            return "lose"

    def _update_streak(self, outcome: str):
        if outcome == "win":
            self.streak = self.streak + 1 if self.streak > 0 else 1
        elif outcome == "lose":
            self.streak = self.streak - 1 if self.streak < 0 else -1
        else:
            self.streak = 0
        self.best_streak = max(self.best_streak, self.streak)

    def play_round(self, player_choice: Choice) -> RoundResult:
        """Play one round. Returns the result."""
        computer_choice = random.choice(list(Choice))
        outcome         = self._determine_outcome(player_choice, computer_choice)

        if outcome == "win":  self.wins   += 1
        elif outcome == "lose": self.losses += 1
        else:                 self.ties   += 1

        self._update_streak(outcome)
        result = RoundResult(self.total_rounds + 1, player_choice, computer_choice, outcome)
        self.rounds.append(result)
        return result

    def format_round(self, result: RoundResult) -> str:
        p = EMOJI[result.player]
        c = EMOJI[result.computer]
        if result.outcome == "win":
            msg    = random.choice(WIN_MSG)
            symbol = "✅"
        elif result.outcome == "lose":
            msg    = random.choice(LOSE_MSG)
            symbol = "❌"
        else:
            msg    = random.choice(TIE_MSG)
            symbol = "🤝"

        streak_str = ""
        if abs(self.streak) >= 2:
            streak_str = f" 🔥 {abs(self.streak)}-streak!"

        return (
            f"  Round {result.round_num:2d}  │  "
            f"You: {p}  vs  CPU: {c}\n"
            f"          │  {symbol} {msg}{streak_str}\n"
            f"          │  Score: {self.wins}W {self.losses}L {self.ties}T "
            f"│ Win rate: {self.win_rate:.0f}%"
        )

    def final_stats(self):
        print(f"\n  {'═'*50}")
        print(f"  🏆  FINAL STATS — {self.player_name}")
        print(f"  {'═'*50}")
        print(f"  Rounds played : {self.total_rounds}")
        print(f"  Wins          : {self.wins}")
        print(f"  Losses        : {self.losses}")
        print(f"  Ties          : {self.ties}")
        print(f"  Win rate      : {self.win_rate:.1f}%")
        print(f"  Best streak   : {self.best_streak}")

        if self.rounds:
            freq = Counter(r.player for r in self.rounds)
            fav  = max(freq, key=freq.get)
            print(f"  Favourite     : {EMOJI[fav]} ({freq[fav]}x)")

        verdict = (
            "🏆 You won overall!" if self.wins > self.losses else
            "🤖 Computer won overall." if self.losses > self.wins else
            "🤝 Overall tie!"
        )
        print(f"  {'─'*50}")
        print(f"  {verdict}")
        print(f"  {'═'*50}")

print("✅ Snake Water Gun game defined.")

✅ Snake Water Gun game defined.


In [2]:
# ---- Demo — simulate 10 rounds with varied player choices ----

random.seed(42)   # fixed seed for reproducibility

game = GameSession("Purvi")

# Simulate player strategy: mix of all choices
player_moves = [
    Choice.SNAKE, Choice.WATER, Choice.GUN,
    Choice.SNAKE, Choice.GUN,   Choice.WATER,
    Choice.GUN,   Choice.SNAKE, Choice.WATER,
    Choice.GUN,
]

print("=" * 52)
print("    🐍💧🔫  SNAKE WATER GUN — 10 ROUNDS")
print("=" * 52)

for move in player_moves:
    result = game.play_round(move)
    print(game.format_round(result))
    print()

game.final_stats()

    🐍💧🔫  SNAKE WATER GUN — 10 ROUNDS
  Round  1  │  You: 🐍 Snake  vs  CPU: 🔫 Gun
          │  ❌ Computer wins!
          │  Score: 0W 1L 0T │ Win rate: 0%

  Round  2  │  You: 💧 Water  vs  CPU: 🐍 Snake
          │  ❌ So close... 🔥 2-streak!
          │  Score: 0W 2L 0T │ Win rate: 0%

  Round  3  │  You: 🔫 Gun  vs  CPU: 🐍 Snake
          │  ✅ Nailed it!
          │  Score: 1W 2L 0T │ Win rate: 33%

  Round  4  │  You: 🐍 Snake  vs  CPU: 🐍 Snake
          │  🤝 Great minds...
          │  Score: 1W 2L 1T │ Win rate: 25%

  Round  5  │  You: 🔫 Gun  vs  CPU: 🔫 Gun
          │  🤝 Great minds...
          │  Score: 1W 2L 2T │ Win rate: 20%

  Round  6  │  You: 💧 Water  vs  CPU: 🔫 Gun
          │  ✅ Unstoppable!
          │  Score: 2W 2L 2T │ Win rate: 33%

  Round  7  │  You: 🔫 Gun  vs  CPU: 🐍 Snake
          │  ✅ You crushed it! 🔥 2-streak!
          │  Score: 3W 2L 2T │ Win rate: 43%

  Round  8  │  You: 🐍 Snake  vs  CPU: 🐍 Snake
          │  🤝 Snap!
          │  Score: 3W 2L 3T │ Win rate: